# 1.4 Persistent CSR Context

## 本节目标

- 区分一次性初始化和稳态迭代
- 理解矩阵、分区和工作缓冲区复用
- 正确解释 cold start、warm kernel、warm total

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
import platform, shutil
print("Python:", platform.python_version())
print("CMake:", shutil.which("cmake"))
print("正式路径：Ascend C FP32 RTC；FP16/BF16/persistent 仍为 Host Prototype")


## 对 GMRES 场景有意义的生命周期

GMRES 会在矩阵 `A` 不变时反复调用 SpMV。`HostPrototypeSpmvContext::initialize_bf16_autotuned()` 完成 Host BF16 转换、候选 partition 测量和缓冲区初始化；`run_bf16()` 仍是 Host Prototype。真实 Device persistent BF16 尚未实现，因此本节不能把该路径计为 NPU。

当前这些动作都由主机容器完成，但“不可变矩阵只准备一次”的生命周期正是未来 Device-resident backend 应保留的设计。

## 计时口径

- `cold_start`：矩阵转换、分区选择和 buffer 初始化
- `warm_kernel`：稳态计算段
- `warm total`：每轮输入准备、计算和输出的合计
- `amortized_runtime`：把一次性成本按重复次数摊销

原 README 明确指出 GMRES 场景应主要比较 warm total，而不是把 cold start 当成每轮成本。

## 课后实践

若一个矩阵只计算一次，persistent 是否仍有优势？若计算 1000 次，应怎样报告 cold/warm？参考答案见 `answer/01.04_answer.md`。